In [ ]:
from __future__ import annotations
from typing import List

import h5py
import numpy as np
from pathlib import Path

In [ ]:
# Gadget IO library for reading Gadget snapshots
# Download from https://www.github.com/masterdesky/glio
try:
    import glio
except ImportError as _err:
    glio = None
    # _GLIO_IMPORT_ERROR = _err

In [ ]:
import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

### I/O with ASCII snapshots

In [ ]:
def load_ascii(files: List[Path], **kwargs):
    '''Load a cosmological snapshot from an ASCII file.'''
    dtype = kwargs.get('dtype', np.float32)
    particleIDs, coordinates, velocities, masses = [], [], [], []
    log.info(f'Reading the input ASCII files ...')
    for path in files:
        log.info(f'Opening ASCII file {path}...')
        data = np.loadtxt(path)
        particleIDs.append(np.array(data[:, 0], dtype=np.uint64))
        coordinates.append(np.array(data[:, 1:4], dtype=dtype))
        velocities.append(np.array(data[:, 4:7], dtype=dtype))
        masses.append(np.array(data[:, 7], dtype=dtype))
    particleIDs = np.concatenate(particleIDs, dtype=np.uint64)
    coordinates = np.concatenate(coordinates, dtype=dtype)
    velocities = np.concatenate(velocities, dtype=dtype)
    masses = np.concatenate(masses, dtype=dtype)
    return particleIDs, coordinates, velocities, masses

In [ ]:
def create_mock_ascii(path: Path, **kwargs):
    '''Create a mock ASCII file for testing purposes.'''
    log.info('Creating mock ASCII file...')
    dtype = kwargs.get('dtype', np.float32)
    data = np.random.rand(100, 8).astype(dtype)  # 100 particles, 8 columns
    np.savetxt(path, data)
    log.info(f'Mock ASCII file created at: {path}')

In [ ]:
path = Path('output', 'snapshot.txt')
create_mock_ascii(path, dtype=np.float32)
ids, coords, vels, masses = load_ascii([path], dtype=np.float32)

### I/O with HDF5 snapshots

In [ ]:
def load_hdf5(files: List[Path], *args, **kwargs):
    '''Load a cosmological snapshot from an HDF5 file.'''
    log.info(f'Reading the input HDF5 files ...')
    part_type = kwargs.get('part_type', 1)
    if not args:
        args = ['ParticleIDs', 'Coordinates', 'Velocities', 'Masses']
    arguments = {ai: [] for ai in args}
    dtypes = {ai: None for ai in args}
    for f in files:
        log.info(f'Opening HDF file {f}...')
        with h5py.File(f, 'r') as hdf:
            for ai in args:
                arguments[ai].append(hdf[f'/PartType{part_type}/{ai}'][:])
                dtypes[ai] = hdf[f'/PartType{part_type}/{ai}'].dtype
            if 'Masses' in args:
                N_part = hdf['/Header'].attrs['NumPart_ThisFile'][part_type]
                mass_part_type = hdf['/Header'].attrs['MassTable'][part_type]
                if np.all(arguments['Masses'] == 0):
                    arguments['Masses'] = np.ones(N_part) * mass_part_type
                if kwargs.get('constant_res', False):
                    arguments['Masses'] *= mass_part_type
    for ai in args:
        arguments[ai] = np.concatenate(arguments[ai], dtype=dtypes[ai])
    return arguments.values()

In [ ]:
def create_mock_hdf5(path: Path, header, **kwargs):
    '''Create a mock HDF5 file for testing purposes.'''
    log.info('Creating mock HDF5 file...')
    part_type = kwargs.get('part_type', 1)
    dtype = kwargs.get('dtype', np.float32)

    # Mock data
    data = np.random.rand(100, 7).astype(dtype)  # 100 particles, 7 columns

    with h5py.File(path, 'w') as hdf_file:
        header_group = hdf_file.create_group('/Header')
        num_part_array = np.zeros(6, dtype=np.uint32)
        num_part_array[part_type] = data.shape[0]
        
        header_group.attrs['NumPart_ThisFile'] = num_part_array
        header_group.attrs['NumPart_Total'] = num_part_array
        header_group.attrs['NumPart_Total_HighWord'] = np.zeros(6, dtype=np.uint32)
        header_group.attrs['MassTable'] = np.zeros(6, dtype=dtype)
        header_group.attrs['Time'] = 1.0 / (header.get('Redshift', 0) + 1.0)
        header_group.attrs['Redshift'] = float(header.get('Redshift', 0.0))
        header_group.attrs['BoxSize'] = float(header.get('BoxSize', 0.0))
        header_group.attrs['NumFilesPerSnapshot'] = header.get('NumFilesPerSnapshot', 1)
        header_group.attrs['Omega0'] = float(header.get('OmegaM', 0.0))
        header_group.attrs['OmegaLambda'] = float(header.get('OmegaL', 0.0))
        header_group.attrs['HubbleParam'] = float(header.get('HubbleParam', 0.0))
        header_group.attrs['Flag_Sfr'] = 0
        header_group.attrs['Flag_Cooling'] = 0
        header_group.attrs['Flag_StellarAge'] = 0
        header_group.attrs['Flag_Metals'] = 0
        header_group.attrs['Flag_Feedback'] = 0
        header_group.attrs['Flag_Entropy_ICs'] = 0

        p_group = hdf_file.create_group(f'/PartType{part_type}')

        if data.shape[1] == 8:
            ID = data[:, 0].astype(np.uint64, copy=False)
        else:
            ID = np.arange(data.shape[0], dtype=np.uint64)
            data = np.concatenate((ID[:, np.newaxis], data), axis=1)
        p_group.create_dataset('ParticleIDs', data=ID)
        p_group.create_dataset('Coordinates', data=data[:, [1, 2, 3]], dtype=dtype)
        p_group.create_dataset('Velocities', data=data[:, [4, 5, 6]], dtype=dtype)
        p_group.create_dataset('Masses', data=data[:, 7], dtype=dtype)

In [ ]:
path = Path('output', 'snapshot.hdf5')

header = {
    'Redshift': 63.0,
    'BoxSize': 100.0,
    'NumFilesPerSnapshot': 1,
    'OmegaM': 0.3,
    'OmegaL': 0.7,
    'HubbleParam': 0.6777
}

create_mock_hdf5(path, header, dtype=np.float32)
ids, coords, vels, masses = load_hdf5([path], dtype=np.float32)

In [ ]:
path = Path('../examples', 'CylindricalGlass_L200_R100_N60k.hdf5')
ids, coords, vels, masses = load_hdf5([path], dtype=np.float32)

### I/O with Gadget snapshots

TODO